# 第 02 章练习

完成下面的练习，结合自己的运行结果作答。


<a id="s-02-06_chapter_test"></a>


## 2.6 OpenMP 扩展性

比较线程数和调度方式，分析误差、耗时与加速比。已完成 2.5 节时，可复用三份线程数 CSV，跳过下面的“构建与线程数比较”，直接运行“补充调度对照”。


### 设置 Host 编译器路径

先完成第 01 章的编译器安装，再运行下面的单元格，为当前 Notebook 设置编译器路径。


In [ ]:
import os
from pathlib import Path

# 若安装在其他位置，只修改这一行。
compiler = Path.home() / "opt/BiShengCompiler-3.2.0-aarch64-linux"
os.environ["PATH"] = f"{compiler}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{compiler}/lib:{compiler}/lib/aarch64-unknown-linux-gnu:" + os.environ.get("LD_LIBRARY_PATH", "")


### 构建与线程数比较（已完成 2.5 节可跳过）

单独做本节练习时，先编译程序，再比较 U1 的 1、2、4 线程。线程数不超过分配的 CPU 核心数。


In [ ]:
%%bash
set -e
cd src/openmp_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_COMPILER=clang++
cmake --build build -j
# 学生修改：线程数列表。资源不足 4 核时，也把下方两种调度的线程数一起改小。
for threads in 1 2 4; do
  OMP_NUM_THREADS=$threads OMP_SCHEDULE=static \
    ./build/bin/spmv_benchmark --matrix U1 --warmup 1 --repeat 3 \
    --csv "results/U1_${threads}t_static.csv"
done


### 补充调度对照

U1 的 4 线程 static 结果已在前面生成。这一格只补跑 U1 的 dynamic，以及 L1 的两种调度。保持相同线程数；下面默认 4 线程。


In [ ]:
%%bash
set -e
cd src/openmp_spmv
OMP_NUM_THREADS=4 OMP_SCHEDULE=dynamic,16 \
  ./build/bin/spmv_benchmark --matrix U1 --warmup 1 --repeat 3 \
  --csv results/U1_4t_dynamic16.csv

# L1：保持矩阵、线程数和重复次数一致，只改变调度策略。
OMP_NUM_THREADS=4 OMP_SCHEDULE=static \
  ./build/bin/spmv_benchmark --matrix L1 --warmup 1 --repeat 3 \
  --csv results/L1_4t_static.csv
OMP_NUM_THREADS=4 OMP_SCHEDULE=dynamic,16 \
  ./build/bin/spmv_benchmark --matrix L1 --warmup 1 --repeat 3 \
  --csv results/L1_4t_dynamic16.csv


### 练习与提交

1. 用一个 CSR 小例子解释 `row_ptr[rows]=nnz`，说明按行并行为什么不需要原子加法。
2. 读取 1/2/4 线程的 CSV，比较耗时；资源足够时再加 8 线程。表中保留实际线程数、串行时间、OpenMP 时间、加速比和误差。
3. 对 U1 和 L1 各比较 static 与 dynamic,16，固定其余参数，结合行长度说明差异。
4. 从 CSV 计算并行效率和平均每线程估算带宽。提出一个能区分访存限制与调度开销的后续实验。

先通过误差检查，再分析性能；表中使用每次运行同时测得的串行基线。

[参考答案](answer/02.06_answer.md)
